# 01 — Data Exploration
## Unified Eye Disease Dataset

**Goal:** explore the dataset, understand splits, visualize samples per class, and analyze class imbalance.

**Assumed layout** (adjust `CONFIG` below to match your setup):
```
data/
 ├── train.csv
 ├── val.csv
 ├── test.csv
 └── images/            # all fundus images referenced by `filename`
```
Each CSV has columns: `filename, dataset, unified_label, original_label`.
If your images are split into `train/`, `val/`, `test/` subfolders instead of one `images/` folder,
just change `IMG_DIR` per split in the loader helper below.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

CONFIG = {
    "DATA_DIR": "/kaggle/working/dataset_extracted/splits",  # folder containing train.csv / val.csv / test.csv
    "IMG_DIR": "/kaggle/working/dataset_extracted/images",   # folder containing the actual image files
    "LABEL_COL": "unified_label",
    "FILENAME_COL": "filename",
}

CLASSES = ["Normal", "Diabetic Retinopathy", "Others", "Glaucoma",
           "Cataract", "Myopia", "AMD", "Hypertension"]


In [ ]:
# --- Load splits ---
train_df = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "train.csv"))
val_df   = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "val.csv"))
test_df  = pd.read_csv(os.path.join(CONFIG["DATA_DIR"], "test.csv"))

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(df)} rows")

train_df.head()

## Summary statistics per class / split

In [ ]:
summary = pd.DataFrame({
    "Train": train_df[CONFIG["LABEL_COL"]].value_counts(),
    "Val":   val_df[CONFIG["LABEL_COL"]].value_counts(),
    "Test":  test_df[CONFIG["LABEL_COL"]].value_counts(),
}).fillna(0).astype(int)
summary["Total"] = summary.sum(axis=1)
summary = summary.reindex(CLASSES)
summary = summary[["Total", "Train", "Val", "Test"]]
summary.loc["TOTAL"] = summary.sum()
summary

## Class distribution plot (required deliverable)

In [ ]:
counts = summary.drop("TOTAL")["Total"].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(counts.index, counts.values, color=sns.color_palette("viridis", len(counts)))
ax.set_title("Class Distribution — Unified Eye Disease Dataset (Total Images)")
ax.set_ylabel("Number of images")
ax.set_xticklabels(counts.index, rotation=35, ha="right")
for b, v in zip(bars, counts.values):
    ax.text(b.get_x() + b.get_width()/2, v + 30, str(v), ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.show()

In [ ]:
# Stacked train/val/test view
split_df = summary.drop("TOTAL")[["Train", "Val", "Test"]]
split_df.plot(kind="bar", stacked=True, figsize=(9, 5), colormap="viridis")
plt.title("Class Distribution by Split")
plt.ylabel("Number of images")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## Class imbalance analysis

- **Majority class:** Normal (4,698 images)
- **Minority class:** Hypertension (88 images) — a **~53:1** imbalance ratio.
- Several classes (Myopia, AMD, Hypertension) have <300 images total, and <30 in val/test —
  this means per-class metrics for those splits will be noisy/high-variance regardless of model quality.

**Handling strategy used later in the modeling notebooks:**
1. **Class-weighted loss** (inverse frequency) for both the scratch CNN and the fine-tuned model — required by the brief.
2. **Stratified sampling already respected** (train/val/test proportions look consistent per class).
3. Optional: light oversampling of the rarest classes (Hypertension, AMD, Myopia) via a `WeightedRandomSampler` /
   `class_weight` dict — implemented as a toggle in `02_scratch_cnn.ipynb`.
4. **Data augmentation** applied more aggressively to minority classes to reduce overfitting on few samples.

In [ ]:
# Class weights (inverse frequency), reused in training notebooks
total = counts.sum()
n_classes = len(counts)
class_weights = {cls: total / (n_classes * cnt) for cls, cnt in counts.items()}
pd.Series(class_weights).sort_values(ascending=False)

## Visualize sample images per class (required deliverable)

In [ ]:
def load_img(row, img_dir=CONFIG["IMG_DIR"]):
    path = os.path.join(img_dir, row[CONFIG["FILENAME_COL"]])
    if not os.path.exists(path):
        return None
    return Image.open(path).convert("RGB")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, cls in zip(axes, CLASSES):
    subset = train_df[train_df[CONFIG["LABEL_COL"]] == cls]
    ax.set_title(f"{cls}\n(n={len(subset)})", fontsize=11)
    ax.axis("off")
    if len(subset) == 0:
        continue
    row = subset.sample(1, random_state=42).iloc[0]
    img = load_img(row)
    if img is not None:
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, "image not found", ha="center", va="center")

plt.suptitle("Sample Fundus Image per Class", fontsize=15)
plt.tight_layout()
plt.savefig("sample_images_per_class.png", dpi=150)
plt.show()

In [ ]:
# Image size / basic quality sanity check across a small random sample
sample = train_df.sample(min(200, len(train_df)), random_state=0)
sizes = []
for _, row in sample.iterrows():
    img = load_img(row)
    if img is not None:
        sizes.append(img.size)

if sizes:
    ws, hs = zip(*sizes)
    print(f"Width  — min:{min(ws)} max:{max(ws)} mean:{np.mean(ws):.0f}")
    print(f"Height — min:{min(hs)} max:{max(hs)} mean:{np.mean(hs):.0f}")
else:
    print("No images found at CONFIG['IMG_DIR'] — update the path before proceeding.")

## Class notes (for the README / report)

| Class | What it is |
|---|---|
| Normal | Healthy retina, no pathology |
| Diabetic Retinopathy | Retinal damage from diabetes (microaneurysms, hemorrhages, exudates) |
| Glaucoma | Optic nerve damage, often from elevated intraocular pressure |
| Cataract | Lens clouding — reduces fundus image clarity/contrast |
| AMD | Age-related Macular Degeneration — macular drusen/atrophy |
| Hypertension | Hypertensive retinopathy — vascular changes from high blood pressure |
| Myopia | Pathologic myopia — fundus changes from high near-sightedness |
| Others | Miscellaneous / mixed pathologies not covered above |

Next: `02_scratch_cnn.ipynb`